# 31 Resonance Management

How mechanical dressing shifts dominant constraints in SiV spin-phonon systems.

**Source context**

- arXiv:2508.13356, *All-mechanical coherence protection and fast control of a silicon vacancy spin qubit*
- Eliza Cornell seminar at CU Boulder
- Seminar outlook slides on dressed-state noise suppression, fast pulses, IDT bandwidth, piezoelectric efficiency, and cavity-compatible operation

## Central result

Mechanical dressing introduces **dressed-state splitting** as a controllable parameter.

Dressed-state splitting shifts the coherence regime by changing which interaction dominates:

- lower splitting → magnetic fluctuations
- intermediate splitting → extended coherence
- higher splitting → orbital-state coupling, thermal phonons, and nonlinear phononics

This notebook organizes the seminar roadmap around that observation.


## 1. Setup

This notebook treats the paper and seminar as a constraint-discovery process.

Guiding question:

> How does dressed-state splitting shift the dominant coherence constraint?

Working statement:

> Dressed-state splitting is the organizing variable connecting coherence protection, phononic loss channels, device engineering, and cavity-compatible operation.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import networkx as nx
import numpy as np
import pandas as pd

FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

plt.rcParams["figure.dpi"] = 120


## 2. Operating-regime table

The seminar suggests a progression of operating regimes.

Each row identifies a layer, a controllable shift, the dominant constraint, and the design response.


In [ ]:
regimes = pd.DataFrame(
    [
        {
            "Layer": "Physics",
            "Operating regime": "Bare SiV spin",
            "Shift / control": "Introduce mechanical dressing",
            "Dominant constraint": "Low-frequency magnetic noise",
            "Design response": "Apply continuous mechanical drive",
        },
        {
            "Layer": "Physics",
            "Operating regime": "Mechanically dressed spin",
            "Shift / control": "Tune dressed-state splitting",
            "Dominant constraint": "Residual magnetic-noise sensitivity at lower splitting",
            "Design response": "Increase splitting into useful coherence regime",
        },
        {
            "Layer": "Dressed-state",
            "Operating regime": "Intermediate dressed-state splitting",
            "Shift / control": "Select operating point",
            "Dominant constraint": "Balancing coherence improvement with new loss channels",
            "Design response": "Optimize splitting and strain environment",
        },
        {
            "Layer": "Dressed-state",
            "Operating regime": "Higher dressed-state splitting",
            "Shift / control": "Avoid over-driving loss channels",
            "Dominant constraint": "Orbital, thermal, and nonlinear phononic effects",
            "Design response": "Study orbital splitting, phonon modes, and nonlinear response",
        },
        {
            "Layer": "Device",
            "Operating regime": "Fast-pulse control",
            "Shift / control": "Shorten pulse duration",
            "Dominant constraint": "IDT bandwidth",
            "Design response": "Reduce IDT finger count and redesign acoustic transducer",
        },
        {
            "Layer": "Device",
            "Operating regime": "Reduced IDT finger count",
            "Shift / control": "Recover drive strength",
            "Dominant constraint": "Piezoelectric efficiency",
            "Design response": "Improve materials platform, e.g. lithium niobate on diamond",
        },
        {
            "Layer": "System",
            "Operating regime": "Cavity-compatible operation",
            "Shift / control": "Integrate protected coherence with resonant exchange",
            "Dominant constraint": "Protected coherence plus useful spin-phonon coupling",
            "Design response": "Combine continuous-wave suppression with a resonant phononic cavity",
        },
    ]
)

regimes


## 3. Coherence-regime sketch

A qualitative model captures the seminar takeaway:

- magnetic-noise sensitivity decreases as dressed-state splitting increases
- orbital / phononic loss channels grow at higher splitting
- useful coherence lives between these regimes

This is a toy schematic, not a fit to experimental data.


In [ ]:
x = np.linspace(0.1, 1.0, 400)

magnetic_noise = np.exp(-4.0 * x)
orbital_phononic_loss = 0.15 + 2.2 * np.maximum(x - 0.55, 0) ** 2
total_constraint = magnetic_noise + orbital_phononic_loss

opt_idx = np.argmin(total_constraint)
x_opt = x[opt_idx]
y_opt = total_constraint[opt_idx]

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(x, magnetic_noise, label="Magnetic-noise constraint")
ax.plot(x, orbital_phononic_loss, label="Orbital / phononic constraint")
ax.plot(x, total_constraint, label="Combined constraint", linewidth=2.5)

ax.axvline(x_opt, linestyle="--", linewidth=1)
ax.scatter([x_opt], [y_opt], s=70, zorder=5)

ax.text(
    x_opt + 0.02,
    y_opt + 0.08,
    "useful operating regime",
    fontsize=10,
)

ax.set_xlabel("Dressed-state splitting (normalized)")
ax.set_ylabel("Relative constraint strength")
ax.set_title("Dressed-State Splitting Shifts the Dominant Constraint")
ax.legend()
ax.grid(True, alpha=0.3)

output = FIGURES / "31_dressed_state_constraint_sketch.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 4. Layered resonance-management roadmap

The main notebook figure separates the roadmap into four layers:

1. **Physics layer** — magnetic noise, mechanical dressing, dressed-state splitting
2. **Dressed-state layer** — coherence regime plus orbital / thermal / nonlinear phononic channels
3. **Device layer** — fast pulses, bandwidth, IDT geometry, piezoelectric efficiency
4. **System layer** — cavity-compatible operation


In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis("off")

bands = [
    ("Physics layer", 6.15, 7.35),
    ("Dressed-state layer", 4.35, 5.75),
    ("Device layer", 2.35, 3.75),
    ("System layer", 0.8, 1.75),
]

for label, y0, y1 in bands:
    rect = patches.FancyBboxPatch(
        (0.25, y0),
        13.5,
        y1 - y0,
        boxstyle="round,pad=0.02",
        linewidth=1,
        edgecolor="0.65",
        facecolor="0.96",
    )
    ax.add_patch(rect)
    ax.text(0.45, (y0 + y1) / 2, label, ha="left", va="center", fontsize=10, fontweight="bold")

def box(x, y, text, width=2.0, height=0.55):
    rect = patches.FancyBboxPatch(
        (x - width / 2, y - height / 2),
        width,
        height,
        boxstyle="round,pad=0.08",
        linewidth=1.4,
        edgecolor="black",
        facecolor="white",
    )
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=10, fontweight="bold")

def arrow(x1, y1, x2, y2):
    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops=dict(arrowstyle="-|>", lw=1.5, shrinkA=8, shrinkB=8),
    )

# Physics layer: parameter before outcome.
box(2.0, 6.75, "Magnetic\nnoise")
box(5.0, 6.75, "Mechanical\ndressing")
box(8.2, 6.75, "Dressed-state\nsplitting", width=2.25)
box(11.4, 6.75, "Coherence\nregime", width=2.0)

arrow(3.05, 6.75, 3.95, 6.75)
arrow(6.1, 6.75, 7.0, 6.75)
arrow(9.35, 6.75, 10.35, 6.75)

# Dressed-state layer
box(3.0, 5.05, "Orbital-state\ncoupling", width=2.2)
box(6.3, 5.05, "Thermal\nphonons", width=1.9)
box(9.6, 5.05, "Nonlinear\nphononics", width=2.1)

arrow(8.0, 6.45, 3.7, 5.35)
arrow(8.2, 6.45, 6.3, 5.35)
arrow(8.4, 6.45, 9.0, 5.35)

# Device layer
box(2.8, 3.05, "Fast\npulses", width=1.8)
box(5.7, 3.05, "Bandwidth", width=1.8)
box(8.6, 3.05, "IDT\ngeometry", width=1.8)
box(11.8, 3.05, "Piezoelectric\nefficiency", width=2.2)

arrow(3.0, 4.75, 2.8, 3.4)
arrow(6.3, 4.75, 3.2, 3.4)
arrow(9.5, 4.75, 3.6, 3.4)

arrow(3.8, 3.05, 4.7, 3.05)
arrow(6.7, 3.05, 7.6, 3.05)
arrow(9.6, 3.05, 10.7, 3.05)

# System layer
box(7.0, 1.28, "Cavity-compatible\noperation", width=2.8)
arrow(11.8, 2.7, 7.8, 1.55)

ax.text(7, 7.75, "Resonance Management Roadmap", ha="center", va="center", fontsize=16, fontweight="bold")
ax.text(
    7,
    7.48,
    "Dressed-state splitting shifts the coherence regime and exposes the next engineering constraints.",
    ha="center",
    va="center",
    fontsize=11,
)

output = FIGURES / "31_layered_resonance_management.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 5. Compact operating-regime figure

This simplified figure is useful for README placement.


In [ ]:
layers = [
    ("Magnetic noise", "bare-spin coherence limit"),
    ("Mechanical dressing", "drive-induced dressed states"),
    ("Dressed-state splitting", "coherence-regime control"),
    ("Orbital / phononic effects", "higher-splitting constraints"),
    ("Bandwidth", "fast-pulse control"),
    ("Piezoelectric efficiency", "device platform"),
    ("Cavity integration", "useful resonant spin-phonon exchange"),
]

fig, ax = plt.subplots(figsize=(9, 8))

for i, (label, note) in enumerate(layers):
    y = len(layers) - i
    ax.text(
        0.5,
        y,
        label,
        ha="center",
        va="center",
        fontsize=13,
        fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.45", facecolor="white", edgecolor="black"),
    )
    ax.text(0.5, y - 0.28, note, ha="center", va="center", fontsize=9)
    if i < len(layers) - 1:
        ax.annotate(
            "",
            xy=(0.5, y - 0.62),
            xytext=(0.5, y - 0.18),
            arrowprops=dict(arrowstyle="-|>", lw=1.5),
        )

ax.set_xlim(0, 1)
ax.set_ylim(0.4, len(layers) + 0.7)
ax.set_title("Operating-Regime Shifts", fontsize=14, pad=18)
ax.axis("off")

output = FIGURES / "31_operating_regimes.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 6. Exploratory graph

This graph preserves branch structure around dressed-state splitting.

It is useful for exploration because it highlights dressed-state splitting as the branch point linking coherence, phononic channels, and device requirements.


In [ ]:
G = nx.DiGraph()

nodes = [
    "Magnetic\nNoise",
    "Mechanical\nDressing",
    "Dressed-State\nSplitting",
    "Coherence\nRegime",
    "Orbital-State\nCoupling",
    "Thermal\nPhonons",
    "Nonlinear\nPhononics",
    "Fast\nPulses",
    "Bandwidth",
    "IDT\nGeometry",
    "Piezoelectric\nEfficiency",
    "Cavity-Compatible\nOperation",
]

edges = [
    ("Magnetic\nNoise", "Mechanical\nDressing"),
    ("Mechanical\nDressing", "Dressed-State\nSplitting"),
    ("Dressed-State\nSplitting", "Coherence\nRegime"),
    ("Dressed-State\nSplitting", "Orbital-State\nCoupling"),
    ("Dressed-State\nSplitting", "Thermal\nPhonons"),
    ("Dressed-State\nSplitting", "Nonlinear\nPhononics"),
    ("Orbital-State\nCoupling", "Fast\nPulses"),
    ("Thermal\nPhonons", "Fast\nPulses"),
    ("Nonlinear\nPhononics", "Fast\nPulses"),
    ("Fast\nPulses", "Bandwidth"),
    ("Bandwidth", "IDT\nGeometry"),
    ("IDT\nGeometry", "Piezoelectric\nEfficiency"),
    ("Piezoelectric\nEfficiency", "Cavity-Compatible\nOperation"),
]

G.add_nodes_from(nodes)
G.add_edges_from(edges)

pos = {
    "Magnetic\nNoise": (0, 5),
    "Mechanical\nDressing": (1.7, 5),
    "Dressed-State\nSplitting": (3.6, 5),
    "Coherence\nRegime": (5.4, 5),
    "Orbital-State\nCoupling": (5.4, 6.2),
    "Thermal\nPhonons": (5.4, 3.8),
    "Nonlinear\nPhononics": (7.2, 5.9),
    "Fast\nPulses": (7.2, 4.1),
    "Bandwidth": (9.0, 4.1),
    "IDT\nGeometry": (10.8, 4.1),
    "Piezoelectric\nEfficiency": (12.6, 4.1),
    "Cavity-Compatible\nOperation": (14.6, 4.1),
}

plt.figure(figsize=(16, 7))

nx.draw_networkx_nodes(G, pos, node_size=2800, linewidths=1.5, edgecolors="black")
nx.draw_networkx_edges(G, pos, arrows=True, arrowstyle="-|>", arrowsize=18, width=1.5, connectionstyle="arc3,rad=0.05")
nx.draw_networkx_labels(G, pos, font_size=9, font_weight="bold")

plt.title(
    "Exploratory Constraint Graph\n"
    "Dressed-State Splitting Organizes the Branch Point",
    fontsize=14,
    pad=20,
)

plt.axis("off")
plt.tight_layout()

output = FIGURES / "31_resonance_management_graph.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 7. Wanted and managed resonances

The design target can be stated as resonance selection:

- preserve useful spin-phonon coupling
- create dressed states through mechanical control
- manage magnetic, orbital, thermal, and nonlinear loss channels
- choose a dressed-state splitting that keeps the useful interaction accessible


In [ ]:
resonances = pd.DataFrame(
    [
        {
            "Interaction": "Spin ↔ mechanical control drive",
            "Role": "wanted",
            "Design use": "Creates dressed states and enables fast control",
        },
        {
            "Interaction": "Spin ↔ phononic cavity",
            "Role": "wanted",
            "Design use": "Supports information exchange and future gates",
        },
        {
            "Interaction": "Spin ↔ low-frequency magnetic noise",
            "Role": "managed",
            "Design use": "Mechanical dressing reduces sensitivity",
        },
        {
            "Interaction": "Spin ↔ orbital-state channels",
            "Role": "managed",
            "Design use": "Static strain and splitting optimization",
        },
        {
            "Interaction": "Spin ↔ thermal phonons",
            "Role": "managed",
            "Design use": "Temperature, splitting, and drive-regime control",
        },
        {
            "Interaction": "Spin ↔ nonlinear phononic effects",
            "Role": "managed",
            "Design use": "Device and drive-regime refinement",
        },
    ]
)

resonances


## 8. Notebook result

Dressed-state splitting is the organizing variable connecting:

- coherence protection
- phononic loss channels
- device engineering
- cavity-compatible operation

The seminar roadmap can therefore be interpreted as a sequence of operating-regime shifts driven by dressed-state splitting and subsequent engineering refinements.


## 9. Next notebook

Notebook 37 should quantify the operating-regime picture.

### Guiding question

How does dressed-state splitting shift the dominant coherence constraint?

### Candidate outputs

- coherence-regime map
- toy optimization model
- low / intermediate / high splitting comparison
- dominant-constraint phase diagram
